# Lesson 26: multilayer perceptron activity

## Notebook set up
### Imports

In [1]:
# Third party imports
import matplotlib.pyplot as plt
import pandas as pd
from pandas.api.types import is_numeric_dtype

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, roc_auc_score, log_loss, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler, OrdinalEncoder, OneHotEncoder, MinMaxScaler

## 1. Data preparation

### 1.1. Load diabetes dataset

In [2]:
diabetes_df = pd.read_csv('https://gperdrizet.github.io/FSA_devops/assets/data/unit3/diabetes_prediction_train.csv')

In [3]:
diabetes_df.head()

,id,age,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,sleep_hours_per_day,screen_time_hours_per_day,bmi,waist_to_hip_ratio,systolic_bp,...,gender,ethnicity,education_level,income_level,smoking_status,employment_status,family_history_diabetes,hypertension_history,cardiovascular_history,diagnosed_diabetes
0,0,31,1,45,7.7,6.8,6.1,33.4,0.93,112,...,Female,Hispanic,Highschool,Lower-Middle,Current,Employed,0,0,0,1.0
1,1,50,2,73,5.7,6.5,5.8,23.8,0.83,120,...,Female,White,Highschool,Upper-Middle,Never,Employed,0,0,0,1.0
2,2,32,3,158,8.5,7.4,9.1,24.1,0.83,95,...,Male,Hispanic,Highschool,Lower-Middle,Never,Retired,0,0,0,0.0
3,3,54,3,77,4.6,7.0,9.2,26.6,0.83,121,...,Female,White,Highschool,Lower-Middle,Current,Employed,0,1,0,1.0
4,4,54,1,55,5.7,6.2,5.1,28.8,0.90,108,...,Male,White,Highschool,Upper-Middle,Never,Retired,0,1,0,1.0


In [4]:
diabetes_df.info()

print(diabetes_df.shape)

<class 'pandas.DataFrame'>
RangeIndex: 700000 entries, 0 to 699999
Data columns (total 26 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   id                                  700000 non-null  int64  
 1   age                                 700000 non-null  int64  
 2   alcohol_consumption_per_week        700000 non-null  int64  
 3   physical_activity_minutes_per_week  700000 non-null  int64  
 4   diet_score                          700000 non-null  float64
 5   sleep_hours_per_day                 700000 non-null  float64
 6   screen_time_hours_per_day           700000 non-null  float64
 7   bmi                                 700000 non-null  float64
 8   waist_to_hip_ratio                  700000 non-null  float64
 9   systolic_bp                         700000 non-null  int64  
 10  diastolic_bp                        700000 non-null  int64  
 11  heart_rate                          7

Define the label and feature columns below. The label is `diabetes` (binary: 0 or 1). Using separate lists for numerical, nominal and ordinal features makes preprocessing easier.

In [5]:
# Define the label

label = "diagnosed_diabetes"
diabetes_df["diabetes"] = diabetes_df[label]
diabetes_df.drop(label,axis=1,inplace=True)
label = "diabetes"
numeric_features = []

# Define numerical, ordinal and nominal features
for col in diabetes_df.columns:
    if is_numeric_dtype(diabetes_df[col]) and col != label and col != "id":
        numeric_features.append(col)

nominal_features = ["gender","ethnicity","smoking_status","employment_status"]
ordinal_features = ["education_level","income_level"]
print(numeric_features)
print(nominal_features)
print(ordinal_features)

# Complete feature list
features = numeric_features + nominal_features + ordinal_features

['age', 'alcohol_consumption_per_week', 'physical_activity_minutes_per_week', 'diet_score', 'sleep_hours_per_day', 'screen_time_hours_per_day', 'bmi', 'waist_to_hip_ratio', 'systolic_bp', 'diastolic_bp', 'heart_rate', 'cholesterol_total', 'hdl_cholesterol', 'ldl_cholesterol', 'triglycerides', 'family_history_diabetes', 'hypertension_history', 'cardiovascular_history']
['gender', 'ethnicity', 'smoking_status', 'employment_status']
['education_level', 'income_level']


In [6]:
# Select the features of interest and the label
diabetes_df = diabetes_df[features + [label]]

In [7]:
# YOUR CODE HERE: implement IQR clipping for numerical features
for col in numeric_features:
    q1 = diabetes_df[col].quantile(0.25)
    q3 = diabetes_df[col].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    print(f'{col} lower: {lower_bound} - upper: {upper_bound}')
    df_cleaned = diabetes_df[(diabetes_df[col] < lower_bound) | (diabetes_df[col] > upper_bound)]


age lower: 18.0 - upper: 82.0
alcohol_consumption_per_week lower: -2.0 - upper: 6.0
physical_activity_minutes_per_week lower: -21.5 - upper: 166.5
diet_score lower: 2.0 - upper: 10.0
sleep_hours_per_day lower: 4.600000000000001 - upper: 9.399999999999999
screen_time_hours_per_day lower: 0.3999999999999986 - upper: 11.600000000000001
bmi lower: 18.049999999999997 - upper: 33.650000000000006
waist_to_hip_ratio lower: 0.7549999999999999 - upper: 0.9550000000000001
systolic_bp lower: 84.0 - upper: 148.0
diastolic_bp lower: 57.5 - upper: 93.5
heart_rate lower: 50.0 - upper: 90.0
cholesterol_total lower: 139.0 - upper: 235.0
hdl_cholesterol lower: 31.5 - upper: 75.5
ldl_cholesterol lower: 48.5 - upper: 156.5
triglycerides lower: 56.5 - upper: 188.5
family_history_diabetes lower: 0.0 - upper: 0.0
hypertension_history lower: 0.0 - upper: 0.0
cardiovascular_history lower: 0.0 - upper: 0.0


In [8]:
print(df_cleaned.shape)

(21227, 25)


### 1.2. Train test split

Use `train_test_split` to split the data into training and testing sets. Use `random_state=315` for reproducibility.

In [9]:
training_df, testing_df = train_test_split(diabetes_df,test_size=0.2,random_state=315)


### 1.3. Preprocess numerical features

#### 1.3.1. Standard scale

Neural networks perform better when features are scaled. Use `StandardScaler` to fit on the training features and transform both training and testing features.

**Hint:** Fit the scaler on `training_df[numerical_features]`, then transform both `training_df[numerical_features]` and `testing_df[numerical_features]`.

In [10]:
feature_scaler = StandardScaler()

feature_scaler.fit(training_df[numeric_features])
scaled_training = pd.DataFrame(feature_scaler.transform(training_df[numeric_features]),columns=numeric_features)
scaled_testing  = pd.DataFrame(feature_scaler.transform(testing_df[numeric_features]),columns=numeric_features)
print(type(scaled_training))

#print(scaled_training)
#print(scaled_testing)


<class 'pandas.DataFrame'>


#### 1.3.2. Clip outliers with IQR method

### 1.4. Preprocess categorical features

#### 1.4.1. Ordinal feature encoding

In [11]:
# YOUR CODE HERE: create and fit OrdinalEncoder, then transform both training and testing data

#### 1.4.2. Nominal feature encoding

In [12]:
# YOUR CODE HERE: create and fit OneHotEncoder, transform features, and concatenate back to dataframes

#### 1.4.3. Update the feature list

In [13]:
# YOUR CODE HERE: update the features list to include encoded features and remove the label

#### 1.4.4. Min/max scale categorical features

In [14]:
# YOUR CODE HERE: create MinMaxScaler with feature_range=(-1, 1), fit on categorical features, and transform

In [15]:
training_df.info()

<class 'pandas.DataFrame'>
Index: 560000 entries, 661870 to 422499
Data columns (total 25 columns):
 #   Column                              Non-Null Count   Dtype  
---  ------                              --------------   -----  
 0   age                                 560000 non-null  int64  
 1   alcohol_consumption_per_week        560000 non-null  int64  
 2   physical_activity_minutes_per_week  560000 non-null  int64  
 3   diet_score                          560000 non-null  float64
 4   sleep_hours_per_day                 560000 non-null  float64
 5   screen_time_hours_per_day           560000 non-null  float64
 6   bmi                                 560000 non-null  float64
 7   waist_to_hip_ratio                  560000 non-null  float64
 8   systolic_bp                         560000 non-null  int64  
 9   diastolic_bp                        560000 non-null  int64  
 10  heart_rate                          560000 non-null  int64  
 11  cholesterol_total                   5

## 2. Logistic regression model

Logistic regression is a linear model for classification. It serves as a good baseline before trying more complex models like neural networks.

### 2.1. Fit

Create a `LogisticRegression` model and fit it on the training data. Use `max_iter=1000` to ensure convergence.

In [16]:
logistic_model = # YOUR CODE HERE
fit_result = # YOUR CODE HERE

SyntaxError: invalid syntax (3363220363.py, line 1)

### 2.2. Test set evaluation

For classification, we can use accuracy, F1 score and/or AUC-ROC (and others) instead of R². Use sklearn's [`metrics`](https://scikit-learn.org/stable/api/sklearn.metrics.html) module .

In [ ]:
logistic_predictions = # YOUR CODE HERE
logistic_accuracy = # YOUR CODE HERE
logistic_f1 = # YOUR CODE HERE
logistic_auc = # YOUR CODE HERE
print(f'Logistic regression accuracy on test set: {logistic_accuracy:.4f}')
print(f'Logistic regression F1 score on test set: {logistic_f1:.4f}')
print(f'Logistic regression AUC-ROC score on test set: {logistic_auc:.4f}')

### 2.3. Performance analysis

For classification, visualize performance using a confusion matrix.

In [ ]:
# YOUR CODE HERE

## 3. Multilayer perceptron (MLP) classifier

Now let's build a neural network classifier using sklearn's [`MLPClassifier`](https://scikit-learn.org/stable/modules/generated/sklearn.neural_network.MLPClassifier.html).

### 3.1. Single epoch training function

Complete the training function below. It should:
1. Split the data into training and validation sets
2. Call `partial_fit` on the model (remember to pass `classes=[0, 1]` on the first call)
3. Record training and validation [`log_loss`](https://scikit-learn.org/stable/modules/generated/sklearn.metrics.log_loss.html) (aka binary cross-entropy) in the history dictionary

**Hint:** Use `model.partial_fit(X, y, classes=[0, 1])` for the first epoch. For subsequent epochs, `partial_fit` remembers the classes.

In [ ]:
def train(model: MLPClassifier, df: pd.DataFrame, training_history: dict, classes: list = None) -> tuple[MLPClassifier, dict]:
    '''Trains sklearn MLP classifier model on given dataframe using validation split.
    Returns the updated model and training history dictionary containing training and
    validation log loss. If classes are not provided, assumes 0 and 1.'''

    global features, label

    df, val_df = train_test_split(df, random_state=315)
    
    # YOUR CODE HERE: call partial_fit on the model
    # If classes is provided, pass it to partial_fit
    
    # YOUR CODE HERE: append training and validation log loss to history
    
    return model, training_history

### 3.2. Model training

Create an `MLPClassifier` with:
- `hidden_layer_sizes=(64, 32)` - two hidden layers
- `activation='relu'` - ReLU activation function
- `learning_rate_init=0.001` - initial learning rate
- `warm_start=True` - keep weights between calls to fit
- `random_state=315` - for reproducibility

Train for 10 epochs using the training function above.

In [ ]:
epochs = 10

training_history = {
    'training_loss': [],
    'validation_loss': []
}

mlp_model = # YOUR CODE HERE: create MLPClassifier

for epoch in range(epochs):

    # YOUR CODE HERE

### 3.3. Learning curves

Plot the training and validation loss over epochs to visualize the learning process.

In [ ]:
# YOUR CODE HERE: plot training and validation loss
# Use plt.plot() for each curve
# Add title, xlabel, ylabel, and legend

### 3.4. Test set evaluation

Evaluate the MLP model on the test set, similar to how you evaluated the logistic regression model.

In [ ]:
mlp_predictions = # YOUR CODE HERE
mlp_accuracy = # YOUR CODE HERE
mlp_f1 = # YOUR CODE HERE
mlp_auc = # YOUR CODE HERE
print(f'MLP accuracy on test set: {mlp_accuracy:.4f}')
print(f'MLP F1 score on test set: {mlp_f1:.4f}')
print(f'MLP AUC-ROC score on test set: {mlp_auc:.4f}')

### 3.5. Performance analysis

Create a confusion matrix for the MLP model predictions.

In [ ]:
# YOUR CODE HERE: create confusion matrix for MLP predictions
# Follow the same pattern as the logistic regression confusion matrix

## 4. Model comparison

Compare the performance of both models side by side.

In [ ]:
print(f'Logistic Regression accuracy on test set: {logistic_accuracy:.4f}')
print(f'MLP accuracy on test set: {mlp_accuracy:.4f}')

Create a side-by-side comparison of the confusion matrices for both models.

In [ ]:
# YOUR CODE HERE